# History from nsepy

In [1]:
# hack to change jupyter directory in notebooks for imports
import os
from pathlib import Path
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
else:
    root = Path.cwd()
os.chdir(root)

# Logger
import logging
import yaml

with open(root / 'config' / 'log.yml', 'r') as f:
    config = yaml.safe_load(f.read())
    logging.config.dictConfig(config)

log = logging.getLogger('ib_log')

In [2]:
# get the symbols
from src.nse.nse import get_nse_syms
df_syms = get_nse_syms()

# convert to scrips with is_index = True for Indexes like NIFTY
scrips = df_syms.set_index('symbol')['secType'].eq('IND').to_dict()

In [3]:
# Gets history if they are old

from pathlib import Path
import pandas as pd
import tqdm

from src.nse.nse import get_nse_hist
from src.support import pickle_age

days_old = 1 # set how many days old
hists_file = root / 'data' / 'master'

hist_age_delta = pickle_age(hists_file)['nse_hists.pkl']
hist_age = hist_age_delta.days + hist_age_delta.hours/24 + hist_age_delta.minutes/24/60

old_hists = hist_age > days_old

if old_hists:
    dfs = [get_nse_hist(s) for s in tqdm.tqdm(df_syms.symbol)] # regenerate
else:
    dfs = [pd.read_pickle(hists_file / 'nse_hists.pkl')]

# assemble the hists
nse_hists = pd.concat(dfs,ignore_index=True)

# store the nse_hists if new
if old_hists:
    data_path = root / 'data' / 'master' / 'nse_hists.pkl'
    nse_hists.to_pickle(data_path)

In [4]:
nse_hists

,symbol,date,open,high,low,close,volume,turnover,series,prev_cls,last,vwap,trades,d_volume,pct_delv
0,BANKNIFTY,2022-03-14,34660.25,35435.10,34625.05,35312.15,193297091,7.589670e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BANKNIFTY,2022-03-15,35467.95,35643.80,34706.05,35022.65,224193669,8.437880e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BANKNIFTY,2022-03-16,35555.75,35806.35,35461.55,35748.25,173798310,5.703970e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BANKNIFTY,2022-03-17,36302.00,36611.95,36261.65,36428.55,181953699,6.709490e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BANKNIFTY,2022-03-21,36545.85,36600.20,35900.20,36018.50,128655959,4.771040e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47115,TORNTPOWER,2023-03-03,518.50,523.20,508.85,510.75,636927,3.272872e+13,EQ,515.60,510.00,513.85,17678.0,199515.0,0.3132
47116,TORNTPOWER,2023-03-06,515.00,525.45,511.05,520.75,475485,2.474885e+13,EQ,510.75,522.20,520.50,14789.0,104441.0,0.2197
47117,TORNTPOWER,2023-03-08,519.00,534.55,517.50,530.75,730588,3.863328e+13,EQ,520.75,529.40,528.80,14854.0,283722.0,0.3883
47118,TORNTPOWER,2023-03-09,530.50,537.15,528.05,533.35,556576,2.969648e+13,EQ,530.75,532.45,533.56,10002.0,255443.0,0.4590
